# Model Monitoring & Data Drift Detection (Evidently)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/11_MLOps_Deployment/model_monitoring_drift_evidently.ipynb)

Models rot silently: user behavior shifts, upstream pipelines change a column, seasonality moves. Data drift = input distribution changes AFTER deployment - accuracy drops long before anyone notices.

Evidently compares a reference window against live traffic and reports which features drifted, statistically.

In [ ]:
!pip install -q evidently scikit-learn pandas numpy

## 1. Simulate reference vs drifted production data

In [ ]:
import numpy as np, pandas as pd

rng = np.random.default_rng(42)
n = 5000

reference = pd.DataFrame({
    "age": rng.integers(18, 80, n),
    "income": rng.lognormal(10.5, 0.4, n),
    "tenure_years": rng.exponential(4, n).round(1),
    "product": rng.choice(["basic", "plus", "pro"], n, p=[.5, .35, .15]),
})

drifted = reference.copy()
drifted["age"] = rng.integers(30, 85, n)                  # older audience
drifted["income"] = drifted["income"] * 1.35              # inflation shift
drifted.loc[:, "product"] = rng.choice(
    ["basic", "plus", "pro"], n, p=[.25, .35, .40])       # mix shift

print(reference.describe().round(1).loc[["mean"]])
print(drifted.describe().round(1).loc[["mean"]])

## 2. Statistical drift check per feature (KS test)

In [ ]:
from scipy import stats

for col in ["age", "income", "tenure_years"]:
    ks, p = stats.ks_2samp(reference[col], drifted[col])
    print(f"{col:<13} KS={ks:.3f}  p={p:.2e}  {'DRIFTED' if p < 0.01 else 'stable'}")

for col in ["product"]:
    a = reference[col].value_counts(normalize=True)
    b = drifted[col].value_counts(normalize=True)
    chi = stats.chi2_contingency(pd.concat([a, b], axis=1).fillna(0).T)[1]
    print(f"{col:<13} chi2 p={chi:.2e}  {'DRIFTED' if chi < 0.01 else 'stable'}")

## 3. Evidently report - the automated version

In [ ]:
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

report = Report(metrics=[DataDriftPreset()])
report.run(current_data=drifted, reference_data=reference)
result = report.as_dict()

summary = result["metrics"][0]["result"]
print("dataset drift:", summary["dataset_drift"])
for feat, info in summary["drift_by_columns"].items():
    print(f"{feat:<13} score={info['drift_score']:.3f}  drift={info['drift_detected']}")

In [ ]:
report.save_html("drift_report.html")
print("saved drift_report.html - open it for charts per feature")

## Monitoring playbook
| Signal | Meaning | Action |
|---|---|---|
| input drift only | world changed, model may adapt | investigate source; schedule retrain |
| prediction drift | outputs shifting | check label distribution / thresholds |
| accuracy drop (delayed labels) | true degradation | urgent retrain + rollback |
| no drift, accuracy fine | healthy | nothing - enjoy it |

Cadence: daily KS checks on top-10 features, weekly Evidently report, alert when >30% features drift. Log predictions alongside inputs at serving time or none of this is possible.